# OpenML · Phase 3 — ZenML pipelines
Orchestrate **ingest → preprocess → train → evaluate → register** on a ZenML stack wired to
**MLflow** (experiment tracker + model registry) and **MinIO** (S3 artifact store).

Turn on the **Pipelines**, **Track**, and **Monitor** stacks in the console (:8080) first.
ZenML makes trackers/deployers swappable — this is what keeps OpenML generic.

In [ ]:
# ZenML needs a 'source root' to resolve flavors from a notebook (interactive
# sessions have no __file__). `zenml init` marks this folder as the root.
!zenml init

# --- register the ZenML stack (idempotent; safe to re-run) ---
!zenml artifact-store register minio --flavor s3 --path=s3://datasets/zenml \
    --client_kwargs='{"endpoint_url": "http://minio:9000"}' 2>/dev/null || echo 'artifact-store minio: exists'
!zenml experiment-tracker register mlflow --flavor mlflow --tracking_uri=http://mlflow:5000 \
    --tracking_username=openml --tracking_password=none 2>/dev/null || echo 'experiment-tracker mlflow: exists'
!zenml model-registry register mlflow --flavor mlflow 2>/dev/null || echo 'model-registry mlflow: exists'
!zenml stack register openml -a minio -e mlflow -r mlflow -o default --set 2>/dev/null || zenml stack set openml
!zenml stack describe

In [ ]:
from typing_extensions import Annotated
from typing import Tuple
import numpy as np, mlflow
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import ClassifierMixin
from sklearn.metrics import accuracy_score
from zenml import pipeline, step
from zenml.client import Client

et = Client().active_stack.experiment_tracker

@step
def ingest() -> Tuple[Annotated[np.ndarray,'X'], Annotated[np.ndarray,'y']]:
    X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, n_classes=3, random_state=42)
    return X, y

@step
def preprocess(X: np.ndarray, y: np.ndarray) -> Tuple[
        Annotated[np.ndarray,'Xtr'], Annotated[np.ndarray,'Xte'],
        Annotated[np.ndarray,'ytr'], Annotated[np.ndarray,'yte']]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    return Xtr, Xte, ytr, yte     # explicit tuple -> 4 named artifacts

@step(experiment_tracker=et.name)
def train(Xtr: np.ndarray, ytr: np.ndarray) -> ClassifierMixin:
    mlflow.sklearn.autolog(registered_model_name='openml-rf')
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(Xtr, ytr)
    return clf

@step(experiment_tracker=et.name)
def evaluate(clf: ClassifierMixin, Xte: np.ndarray, yte: np.ndarray) -> float:
    acc = float(accuracy_score(yte, clf.predict(Xte)))
    mlflow.log_metric('test_accuracy', acc)
    print('test_accuracy', acc)
    return acc

@pipeline
def openml_pipeline():
    X, y = ingest()
    Xtr, Xte, ytr, yte = preprocess(X, y)
    clf = train(Xtr, ytr)
    evaluate(clf, Xte, yte)

In [ ]:
run = openml_pipeline()
print('\nDone — open the run in the ZenML dashboard (:8237)')

### Where to look
* **ZenML** http://localhost:8237 → Pipelines → `openml_pipeline` — the DAG, steps, artifacts, lineage
* **MLflow** http://localhost:5000 — the run's metrics + registered model `openml-rf`
* **MinIO** http://localhost:9001 — `datasets/zenml/…` step artifacts (content-addressed)

**Pluggability:** swap the tracker/registry/deployer by registering a different ZenML
component (e.g. `--flavor wandb`, or a BentoML/Seldon deployer in Phase 5) and re-running —
no pipeline code changes.